# Trabajo Nº1 – Clasificación de Perros y Gatos con CNN y Transfer Learning

## Parte 1 · Construcción de una CNN base

En esta nueva versión del cuaderno vamos a seguir la metodología del profesor para desarrollar **únicamente la primera parte del trabajo: la CNN creada desde cero**. Nos centraremos en comprender cada paso y en mantener el dataset original de 1000 gatos y 1000 perros sin alterarlo.  Más adelante se podrá añadir el bloque de *transfer learning* en la misma libreta.

In [ ]:
# Imports básicos (señalados también en los apuntes del profesor)
import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Configuración general para tener resultados reproducibles y mensajes claros
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"TensorFlow: {tf.__version__}")

## Carga del dataset (exactamente 2000 imágenes)

El material del enunciado incluye la carpeta `Cats_Dogs` con dos subcarpetas (`cat/` y `dog/`). Cada una contiene 1000 imágenes en color.  
Para seguir el flujo visto en clase:

1. Coloca la carpeta `Cats_Dogs` en el mismo directorio que este notebook (sin borrar ni añadir imágenes).
2. Usaremos `image_dataset_from_directory` para crear automáticamente los *batches* de entrenamiento y validación con una división 80/20.
3. No se cambia el tamaño del dataset; únicamente se reescala cada imagen al tamaño requerido por la red.

> ⚠️ Si la carpeta no está descomprimida en el entorno actual se generará un error indicando que no se encontró el dataset.

In [ ]:
# Configuración de rutas y parámetros principales
DATASET_DIR = pathlib.Path("Cats_Dogs")
IMAGE_SIZE = (180, 180)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        "No se encontró la carpeta 'Cats_Dogs'. Coloca el dataset junto al notebook antes de continuar."
    )

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    shuffle=True,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    shuffle=True,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print(f"Clases detectadas: {class_names}")

# Mejora de rendimiento siguiendo la recomendación del profesor (cache + prefetch)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Exploración rápida de las imágenes

Siempre es buena idea revisar algunas muestras para asegurarnos de que los datos se están cargando correctamente y de que las etiquetas coinciden.

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()

## Definición de la CNN

La arquitectura sigue el estilo visto en los apuntes:

- Bloque de aumento de datos con volteo horizontal y rotación ligera.
- Reescalado de los píxeles a `[0, 1]`.
- Varias capas `Conv2D` + `MaxPooling2D` con activación ReLU.
- `Dropout` para evitar sobreajuste y una capa densa final con activación sigmoide (clasificación binaria).

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.05, seed=SEED),
    ],
    name="data_augmentation",
)

model = keras.Sequential(
    [
        keras.Input(shape=(*IMAGE_SIZE, 3)),
        data_augmentation,
        layers.Rescaling(1.0 / 255),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="cnn_basica_cyd",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Entrenamiento con *callbacks*

Se utilizan los mismos mecanismos explicados por el profesor:

- `EarlyStopping` para frenar el entrenamiento cuando la validación deje de mejorar.
- `ReduceLROnPlateau` para bajar la tasa de aprendizaje si la pérdida se estanca.
- `ModelCheckpoint` para guardar el mejor modelo de la CNN (lo reutilizaremos para generar el CSV).

In [ ]:
CHECKPOINT_DIR = pathlib.Path("model_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "cnn_basica.keras"

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks,
    verbose=1,
)

history_df = pd.DataFrame(history.history)
history_df.tail()

## Evaluación y análisis rápido

Cargamos el mejor checkpoint para medir el desempeño final sobre la validación y graficamos la evolución de la pérdida y la exactitud.  Un valor cercano al 75 %–80 % ya supera ampliamente la *baseline* indicada en el README.

In [ ]:
best_model = keras.models.load_model(CHECKPOINT_PATH)
val_loss, val_acc = best_model.evaluate(val_ds, verbose=0)
print(f"Exactitud en validación: {val_acc:.3f}")
print(f"Pérdida en validación: {val_loss:.3f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df["loss"], label="Entrenamiento")
plt.plot(history_df["val_loss"], label="Validación")
plt.title("Pérdida")
plt.xlabel("Época")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_df["accuracy"], label="Entrenamiento")
plt.plot(history_df["val_accuracy"], label="Validación")
plt.title("Exactitud")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.show()

## Generación del CSV de predicciones

Este bloque se mantiene listo para la entrega final.  Solo debes asegurarte de tener una carpeta `Cats_Dogs_Test` (o la que definas) con las imágenes sin etiqueta que evaluará el jurado.  
El nombre de archivo generado es `predicciones_cnn_basica.csv` y sigue el formato pedido en el enunciado.

In [ ]:
# Generación del CSV de predicciones
def generar_csv_predicciones(directorio_predicciones):
    directorio_predicciones = pathlib.Path(directorio_predicciones)
    if not directorio_predicciones.exists():
        raise FileNotFoundError(
            f"No se encontró el directorio de predicciones: {directorio_predicciones}"
        )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        directorio_predicciones,
        labels=None,
        shuffle=False,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
    )

    best_model = keras.models.load_model(CHECKPOINT_PATH)
    probs = best_model.predict(test_ds).flatten()
    preds = (probs >= 0.5).astype(int)

    image_ids = [pathlib.Path(path).stem for path in test_ds.file_paths]
    submission = pd.DataFrame({"id": image_ids, "label": preds})
    submission = submission.sort_values("id").reset_index(drop=True)
    submission.to_csv("predicciones_cnn_basica.csv", index=False)
    print("Archivo guardado: predicciones_cnn_basica.csv")

# Ejemplo de uso (descomentar cuando exista la carpeta con imágenes nuevas)
# generar_csv_predicciones("Cats_Dogs_Test")